# BERT Model Evaluation on Unseen Test Data

This notebook evaluates your trained BERT model on a completely unseen test dataset with 65,698 articles.

**Goal:** Verify that the model generalizes well to new data beyond the training set.

## Part 1: Load Test Data

In [21]:
import bert_utils
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [22]:
test_path = 'test_data/dataset.csv'
test_df = pd.read_csv(test_path)
print(f"✓ Loaded {len(test_df):,} test articles")
    
    
test_df.head()

✓ Loaded 65,698 test articles


,title,text,unreliable,author
0,WARNING: A Pivotal Moment For The Stock Market...,WARNING: A Pivotal Moment For The Stock Market...,1,Anonymous Coward (UID 72071746)
1,"Trump, top defense officials, discuss North Ko...",WASHINGTON - U.S. President Donald Trump met ...,0,Unknown
2,British civil servants' union calls nationwide...,LONDON - British civil servants will vote nex...,0,Unknown
3,A**hole Of The Day – Michele Bachmann: Muslim...,Michele Bachmann has been pretty quiet since l...,1,Unknown
4,’Deport Fat People’ Posters Appear At CU Bould...,Posters calling on Donald Trump to “Deport Fat...,0,Lucas Nolan


## Preprocessing Data

In [30]:
print(f"Original dataset size: {len(test_df):,}")
test_df_clean = test_df.dropna(subset=['text', 'unreliable']).copy()
test_df_clean = test_df_clean[test_df_clean['text'].fillna('').str.strip() != ''].copy()
test_texts = test_df_clean['text'].tolist()
test_labels_raw = test_df_clean['unreliable'].tolist()
test_labels = [1 - x for x in test_labels_raw]

print(f"\nTest texts prepared: {len(test_texts):,}")
print(f"Test labels prepared: {len(test_labels):,}")

fake_count = sum(1 for x in test_labels if x == 0)
real_count = sum(1 for x in test_labels if x == 1)

print(f"\nTest Data Distribution (after cleaning and label flip):")
print(f"  Fake articles (model label=0): {fake_count:,}")
print(f"  Real articles (model label=1): {real_count:,}")

Original dataset size: 65,698

Test texts prepared: 64,951
Test labels prepared: 64,951

Test Data Distribution (after cleaning and label flip):
  Fake articles (model label=0): 33,148
  Real articles (model label=1): 31,803


## Part 2: Load Trained BERT Model

In [34]:
model, tokenizer = bert_utils.load_model('models/bert_fake_news')
print(f"Model: {type(model).__name__}")
print(f"Tokenizer: {type(tokenizer).__name__}")

INFO:bert_utils:MacBook GPU (MPS) detected
INFO:bert_utils:Model loaded from models/bert_fake_news


Model: BertForSequenceClassification
Tokenizer: BertTokenizer


## Part 3: Evaluate on Unseen Test Data

In [38]:
results = bert_utils.evaluate_bert_model(
    model,
    tokenizer,
    test_texts,
    test_labels,
    batch_size=32
)

INFO:bert_utils:MacBook GPU (MPS) detected
INFO:bert_utils:Accuracy:  0.8594                                                                                                                               
INFO:bert_utils:Precision: 0.8766
INFO:bert_utils:Recall:    0.8296
INFO:bert_utils:F1 Score:  0.8525
INFO:bert_utils:ROC-AUC:   0.9343


## Part 4: Display Results

In [39]:
print(f"\nTotal Test Samples: {len(test_labels):,}")
print(f"\nPerformance Metrics:")
print(f"  Accuracy:  {results['accuracy']:.4f}")
print(f"  Precision: {results['precision']:.4f}")
print(f"  Recall:    {results['recall']:.4f}")
print(f"  F1 Score:  {results['f1']:.4f}")
print(f"  ROC-AUC:   {results['roc_auc']:.4f}")

print(f"\n{'='*70}")


Total Test Samples: 64,951

Performance Metrics:
  Accuracy:  0.8594
  Precision: 0.8766
  Recall:    0.8296
  F1 Score:  0.8525
  ROC-AUC:   0.9343



In [40]:
# Confusion matrix analysis
cm = results['confusion_matrix']

print(f"\nConfusion Matrix:")
print(f"  True Negatives (Fake correctly identified):   {cm[0, 0]:6,}")
print(f"  False Positives (Fake marked as Real):        {cm[0, 1]:6,}")
print(f"  False Negatives (Real marked as Fake):        {cm[1, 0]:6,}")
print(f"  True Positives (Real correctly identified):   {cm[1, 1]:6,}")

total_errors = cm[0, 1] + cm[1, 0]
error_rate = total_errors / len(test_labels)
print(f"\nTotal Misclassifications: {total_errors:,} out of {len(test_labels):,} ({error_rate*100:.3f}%)")


Confusion Matrix:
  True Negatives (Fake correctly identified):   29,435
  False Positives (Fake marked as Real):         3,713
  False Negatives (Real marked as Fake):         5,418
  True Positives (Real correctly identified):   26,385

Total Misclassifications: 9,131 out of 64,951 (14.058%)


In [41]:
# Comparison with training results

print(f"{'Dataset':<20} | {'Samples':<15} | {'Accuracy':<15}")
print("-" * 55)
print(f"{'Training':<20} | {'31,422':<15} | {'N/A (training only)':<15}")
print(f"{'Validation':<20} | {'6,733':<15} | {'99.88%':<15}")
print(f"{'Test (Training Set)':<20} | {'6,734':<15} | {'99.91%':<15}")
accuracy_str = f"{results['accuracy']*100:.2f}%"
print(f"{'Unseen Test Data':<20} | {'65,698':<15} | {accuracy_str:<15}")
print(f"{'='*70}")

Dataset              | Samples         | Accuracy       
-------------------------------------------------------
Training             | 31,422          | N/A (training only)
Validation           | 6,733           | 99.88%         
Test (Training Set)  | 6,734           | 99.91%         
Unseen Test Data     | 65,698          | 85.94%         


## Part 5: Summary & Interpretation

In [42]:
# Save results to JSON
results_file = 'unseen_test_evaluation_results.json'

output_results = {
    'timestamp': datetime.now().isoformat(),
    'dataset': 'unseen_test_data (65,698 articles)',
    'test_samples': len(test_labels),
    'fake_samples': fake_count,
    'real_samples': real_count,
    'label_mapping': {
        'model_convention': 'label=0 (FAKE), label=1 (REAL)',
        'test_data_convention': 'unreliable=0 (REAL), unreliable=1 (FAKE)',
        'note': 'Test labels were flipped before evaluation to match model convention'
    },
    'metrics': {
        'accuracy': float(results['accuracy']),
        'precision': float(results['precision']),
        'recall': float(results['recall']),
        'f1': float(results['f1']),
        'roc_auc': float(results['roc_auc'])
    },
    'confusion_matrix': {
        'true_negatives': int(cm[0, 0]),
        'false_positives': int(cm[0, 1]),
        'false_negatives': int(cm[1, 0]),
        'true_positives': int(cm[1, 1])
    },
    'model_path': 'models/bert_fake_news/'
}

with open(results_file, 'w') as f:
    json.dump(output_results, f, indent=2)

print(f"\n✓ Results saved to {results_file}")
print(f"✓ Label mapping information included in results file")
print(f"\nYou can view the results file to share or archive the evaluation metrics.")


✓ Results saved to unseen_test_evaluation_results.json
✓ Label mapping information included in results file

You can view the results file to share or archive the evaluation metrics.
